In [1]:
"""
Ce script construit un modèle pour classifier les erreurs médicamenteuses en "graves" ou "non graves"
en se basant sur les commentaires des pharmaciens.

Préparé par Anissa OUAREM
"""

# Étape 1 : Importer les bibliothèques nécessaires
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re

# --- Début de la configuration ---

# Étape 2 : Charger les données depuis le fichier CSV
# Assurez-vous d'avoir téléversé le fichier "data_defi3.csv" dans votre environnement Google Colab
file_path = 'data_defi3.csv'
# On lit le fichier en spécifiant le séparateur ";"
try:
    data = pd.read_csv(file_path, sep=';')
except FileNotFoundError:
    print(f"Erreur : Le fichier '{file_path}' n'a pas été trouvé. Veuillez vous assurer qu'il est correctement chargé.")
    exit()


# Étape 3 : Préparer les données
# Nettoyage de base : supprimer les lignes sans commentaire ou sans classe
data.dropna(subset=['Avis.Pharmaceutique', 'PLT'], inplace=True)

# Déterminer si l'erreur est "grave"
# Une erreur est considérée comme grave si la classe commence par 4, 5, 6.3 ou 6.4
# On convertit la colonne en type string pour assurer une comparaison correcte
data['PLT_str'] = data['PLT'].astype(str).str.replace(',', '.')

# Liste des classes considérées comme graves
serious_classes = ['4', '5', '6.3', '6.4']

def is_serious(plt_code):
    """
    Fonction pour vérifier si une classe est grave.
    On recherche une correspondance au début de la chaîne de caractères.
    """
    for s_class in serious_classes:
        if plt_code.startswith(s_class):
            return 1 # 1 signifie "grave"
    return 0 # 0 signifie "non grave"

# Appliquer la fonction pour créer la colonne cible
data['is_grave'] = data['PLT_str'].apply(is_serious)


# Définir les variables d'entrée (X) et la cible (y)
X = data['Avis.Pharmaceutique'] # Les commentaires
y = data['is_grave']             # 1 si grave, 0 sinon


# --- Construction du modèle ---

# Étape 4 : Diviser les données en ensembles d'entraînement et de test
# 80% pour l'entraînement et 20% pour le test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Étape 5 : Transformer le texte en vecteurs numériques (Vectorisation)
# On utilise TfidfVectorizer pour convertir le texte en nombres que le modèle peut comprendre
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# Étape 6 : Entraîner le modèle de classification
# On utilise la Régression Logistique, un modèle simple et efficace pour la classification de texte
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_vec, y_train)


# --- Évaluation du modèle ---

# Étape 7 : Prédiction et évaluation des performances
y_pred = model.predict(X_test_vec)

# Afficher le rapport de performance (précision, rappel, F1-score)
print("--- Rapport de performance du modèle de classification binaire (grave / non grave) ---")
print(classification_report(y_test, y_pred, target_names=['Non grave (0)', 'Grave (1)']))


# --- Exemple d'utilisation ---
print("\n--- Test du modèle sur un nouveau commentaire ---")
new_comment = "Refusé pour le motif suivant : surdosage"
new_comment_vec = vectorizer.transform([new_comment])
prediction = model.predict(new_comment_vec)
prediction_proba = model.predict_proba(new_comment_vec)

print(f"Commentaire : '{new_comment}'")
if prediction[0] == 1:
    print("Résultat : Grave")
else:
    print("Résultat : Non grave")
print(f"Probabilité (Non grave) : {prediction_proba[0][0]:.2f}, Probabilité (Grave) : {prediction_proba[0][1]:.2f}")


# --- Étape 8 : Exporter les prédictions vers un fichier CSV ---
# Selon les consignes, vous devez soumettre les prédictions sur un jeu de données de validation.
# Ici, nous utilisons notre jeu de test (X_test) comme exemple pour la démonstration.
# Vous devrez adapter cette partie pour charger et prédire sur le vrai fichier de validation.

# Créer un DataFrame avec les commentaires de test et les prédictions du modèle
results_df = pd.DataFrame({
    'Avis.Pharmaceutique': X_test,
    'prediction_grave': y_pred
})

# Sauvegarder le DataFrame dans un fichier CSV
output_filename = 'predictions_grave.csv'
# On utilise sep=';' pour la compatibilité avec Excel en France et utf-8-sig pour un encodage correct
results_df.to_csv(output_filename, index=False, sep=';', encoding='utf-8-sig')

print(f"\n--- Exportation terminée ---")
print(f"Les prédictions ont été sauvegardées dans le fichier : '{output_filename}'")

--- Rapport de performance du modèle de classification binaire (grave / non grave) ---
               precision    recall  f1-score   support

Non grave (0)       0.97      0.90      0.93      3723
    Grave (1)       0.68      0.89      0.77       906

     accuracy                           0.90      4629
    macro avg       0.82      0.90      0.85      4629
 weighted avg       0.91      0.90      0.90      4629


--- Test du modèle sur un nouveau commentaire ---
Commentaire : 'Refusé pour le motif suivant : surdosage'
Résultat : Grave
Probabilité (Non grave) : 0.05, Probabilité (Grave) : 0.95

--- Exportation terminée ---
Les prédictions ont été sauvegardées dans le fichier : 'predictions_grave.csv'
